# 제조 음성 ASR 심사 실행 노트북

## Goal

승인된 제조 음성과 검수 전사가 준비된 뒤 모델 비교, 사람 선택, 선택 모델 LoRA, 양자화, 고정 Test와 심사 증빙 점검을 순서대로 실행합니다.

**보안:** 카메라·마이크·패스키를 사용하지 않습니다. 실제 음성은 GitHub에 올리지 않고 승인된 비공개 Drive 경로만 사용합니다.

In [ ]:
GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
GITHUB_BRANCH = "codex/whisper-benchmark-quantization"  # 병합 후 main
PROJECT_DIR = "/content/AIAS"
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"
PRIVATE_ROOT = f"{DRIVE_ROOT}/data/private/manufacturing"
CONFIG = "configs/manufacturing_private_template.yaml"
MODEL_MATRIX = "configs/benchmarks/manufacturing_whisper_models_template.yaml"
QUANTIZATION_SPEC = "configs/quantization/manufacturing_whisper_quantization_template.yaml"
BENCHMARK_ID = "manufacturing-whisper-model-benchmark-v1"
QUANTIZATION_ID = "manufacturing-whisper-quantization-v1"

## Setup

### 1. GPU와 비공개 Drive 연결

In [ ]:
!nvidia-smi
from google.colab import drive

drive.mount("/content/drive")

### 2. GitHub 코드 동기화

In [ ]:
import os, subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            GITHUB_BRANCH,
            "--single-branch",
            GITHUB_REPO_URL,
            PROJECT_DIR,
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"], check=True
    )
os.chdir(PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

### 3. 의존성 설치

In [ ]:
%pip uninstall -y torchao
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"
import sys


def run_aias(*args):
    command = [sys.executable, "-m", "aias_specialist.cli", *args]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

### 4. 승인·manifest·사람 검토 양식 준비

기존 파일은 덮어쓰지 않습니다. 생성된 세 파일을 실제 값으로 채우고 음성을 `PRIVATE_ROOT/audio/` 아래에 업로드합니다.

In [ ]:
from pathlib import Path
import shutil

private_root = Path(PRIVATE_ROOT)
(private_root / "audio").mkdir(parents=True, exist_ok=True)
templates = {
    Path("data/templates/manufacturing_manifest_template.csv"): private_root / "manifest.csv",
    Path("data/templates/data_approval_template.yaml"): private_root / "data_approval.yaml",
    Path("data/templates/human_review_signoff_template.yaml"): private_root
    / "human_review_signoff.yaml",
    Path("configs/assessment/acceptance_criteria_template.yaml"): private_root
    / "acceptance_criteria.yaml",
}
for source, destination in templates.items():
    if not destination.exists():
        shutil.copy2(source, destination)
        print("Created:", destination)
    else:
        print("Preserved existing:", destination)

## Checks

### 5. 데이터 전 준비도 점검

음성과 승인정보가 아직 없으면 `not_ready`가 정상입니다.

In [ ]:
run_aias(
    "assessment-audit",
    "--config",
    CONFIG,
    "--output-dir",
    f"{DRIVE_ROOT}/reports/assessment_readiness",
)

## Steps

아래부터는 승인 문서, manifest, 음성과 전사 검수가 완료된 뒤 실행합니다.

### 6. 모델 비교

In [ ]:
import pandas as pd

run_aias("benchmark-models", "--matrix", MODEL_MATRIX)
benchmark_dir = Path(DRIVE_ROOT) / "artifacts/benchmarks" / BENCHMARK_ID
display(pd.read_csv(benchmark_dir / "benchmark_comparison.csv"))

### 7. 모델 선택

In [ ]:
SELECTED_MODEL = "small"
REVIEWER = "TO_BE_COMPLETED"
MODEL_REASON = "TO_BE_COMPLETED: 정확도·속도·메모리·거버넌스 근거"

run_aias(
    "select-model",
    "--benchmark-dir",
    str(benchmark_dir),
    "--model-id",
    SELECTED_MODEL,
    "--reviewer",
    REVIEWER,
    "--reason",
    MODEL_REASON,
)
model_selection = benchmark_dir / "model_selection.yaml"

### 8. 선택 모델 LoRA

In [ ]:
run_aias(
    "train-selected-whisper",
    "--selection",
    str(model_selection),
    "--config",
    CONFIG,
)

### 9. 양자화 비교 및 선택

In [ ]:
run_aias(
    "quantization-sweep",
    "--spec",
    QUANTIZATION_SPEC,
    "--selection",
    str(model_selection),
)
quantization_dir = Path(DRIVE_ROOT) / "artifacts/quantization" / QUANTIZATION_ID
display(pd.read_csv(quantization_dir / "quantization_comparison.csv"))

In [ ]:
SELECTED_VARIANT = "float16"
QUANTIZATION_REASON = "TO_BE_COMPLETED: 정확도 손실·속도·메모리 근거"

run_aias(
    "select-quantization",
    "--quantization-dir",
    str(quantization_dir),
    "--variant-id",
    SELECTED_VARIANT,
    "--reviewer",
    REVIEWER,
    "--reason",
    QUANTIZATION_REASON,
)
quantization_selection = quantization_dir / "quantization_selection.yaml"

### 10. 고정 Test 최종평가

In [ ]:
run_aias(
    "finalize-evaluation",
    "--selection",
    str(quantization_selection),
    "--config",
    CONFIG,
)

## Next Steps

최종 보고서와 오류 샘플을 사람이 검수하고 `human_review_signoff.yaml`을 완료한 뒤 마지막 점검을 실행합니다.

In [ ]:
run_aias(
    "assessment-audit",
    "--config",
    CONFIG,
    "--output-dir",
    f"{DRIVE_ROOT}/reports/assessment_readiness",
    "--fail-on-blocker",
)